# Voice cloning tiếng Việt trên Colab

Notebook này tạo file MP3 mới dựa trên giọng trong `2.mp3`. Chỉ dùng với giọng của chính bạn hoặc khi bạn có quyền/sự đồng ý rõ ràng từ người sở hữu giọng nói.

**Cách chạy:** Runtime > Change runtime type > GPU, sau đó chạy lần lượt từng cell. Khi được hỏi upload file, chọn đúng `2.mp3`.


Repo: https://github.com/OpenBMB/VoxCPM

Notebook này dùng VoxCPM2. Model hỗ trợ tiếng Việt native và có 2 kiểu clone: `reference_wav_path` để clone nhanh, hoặc `prompt_wav_path + prompt_text + reference_wav_path` để tăng độ giống giọng. Notebook tự transcribe `2.mp3` bằng Whisper để bạn chỉ cần upload audio.

In [ ]:
TEXT_TO_SPEAK = """Xin chào, đây là bản thử nghiệm nhân bản giọng nói tiếng Việt. Nếu đoạn âm thanh tham chiếu rõ ràng, ít nhiễu và chỉ có một người nói, kết quả sẽ giống giọng gốc hơn."""
STYLE = "tu_nhien"  # dùng cho một số model: tu_nhien, tin_tuc, doc_truyen
REFERENCE_MP3 = "/content/2.mp3"
REFERENCE_WAV = "/content/ref.wav"

OUTPUT_WAV = "/content/voxcpm2_output.wav"
OUTPUT_TRIMMED_WAV = "/content/voxcpm2_output_trimmed.wav"
OUTPUT_MP3 = "/content/voxcpm2_output.mp3"
USE_ULTIMATE_CLONING = False  # False giúp audio chỉ đọc TEXT_TO_SPEAK, ít bị lặp/thêm chữ từ prompt.
CONTROL_STYLE = ""  # Để trống để model không đọc nhầm style instruction thành lời nói.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U voxcpm soundfile faster-whisper

In [ ]:
from google.colab import files
import os, shutil, subprocess, textwrap

uploaded = files.upload()
uploaded_name = "2.mp3" if "2.mp3" in uploaded else next(iter(uploaded))
uploaded_path = os.path.abspath(uploaded_name)
if uploaded_path != REFERENCE_MP3:
    shutil.move(uploaded_path, REFERENCE_MP3)

assert os.path.exists(REFERENCE_MP3), "Không tìm thấy /content/2.mp3"

# Cắt 12 giây đầu, mono 24kHz. Nếu file gốc có đoạn im lặng đầu, hãy đổi -ss 0 thành vị trí bắt đầu giọng nói.
subprocess.run([
    "ffmpeg", "-y", "-i", REFERENCE_MP3,
    "-ss", "0", "-t", "12",
    "-ar", "24000", "-ac", "1",
    REFERENCE_WAV
], check=True)

print("Reference WAV:", REFERENCE_WAV)

In [ ]:
import torch, os
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
asr = WhisperModel("small", device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language="vi", vad_filter=True)
REF_TEXT = " ".join(seg.text.strip() for seg in segments).strip()
print("REF_TEXT =", REF_TEXT)
if not REF_TEXT:
    raise RuntimeError("Whisper không nhận ra transcript từ 2.mp3. Hãy dùng đoạn ref rõ tiếng hơn.")

In [ ]:
from voxcpm import VoxCPM
import soundfile as sf
import subprocess, os, torch, gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=False,
)

text = TEXT_TO_SPEAK.strip()
if CONTROL_STYLE.strip():
    text = CONTROL_STYLE.strip() + " " + text

if USE_ULTIMATE_CLONING and REF_TEXT.strip():
    wav = model.generate(
        text=text,
        prompt_wav_path=REFERENCE_WAV,
        prompt_text=REF_TEXT,
        reference_wav_path=REFERENCE_WAV,
        cfg_value=2.0,
        inference_timesteps=10,
    )
else:
    wav = model.generate(
        text=text,
        reference_wav_path=REFERENCE_WAV,
        cfg_value=2.0,
        inference_timesteps=10,
    )

sf.write(OUTPUT_WAV, wav, model.tts_model.sample_rate)

# Trim silence nhẹ ở đầu/cuối; không cắt nội dung nói, chỉ dọn khoảng lặng do model sinh ra.
subprocess.run([
    "ffmpeg", "-y", "-i", OUTPUT_WAV,
    "-af", "silenceremove=start_periods=1:start_duration=0.15:start_threshold=-45dB:stop_periods=1:stop_duration=0.35:stop_threshold=-45dB",
    OUTPUT_TRIMMED_WAV,
], check=True)

subprocess.run(["ffmpeg", "-y", "-i", OUTPUT_TRIMMED_WAV, "-codec:a", "libmp3lame", "-q:a", "2", OUTPUT_MP3], check=True)
print("Done:", OUTPUT_MP3)

In [ ]:
from google.colab import files
files.download(OUTPUT_MP3)